# Ręczna klasyczna dekompozycja addytywna szeregu czasowego

Ten notebook pokazuje ręczną implementację klasycznej dekompozycji addytywnej:

$$Y_t = T_t + S_t + R_t$$

Obsługiwane przypadki:
- okno nieparzyste → zwykła średnia ruchoma MA,
- okno parzyste → centered moving average, czyli CMA.


## 1. Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Mini-biblioteka do dekompozycji

In [ ]:
def calculate_trend(series, window):
    """
    Wyznacza trend:
    - dla okna nieparzystego: centered moving average,
    - dla okna parzystego: CMA, czyli centered moving average z dwóch MA.
    """
    
    series = pd.Series(series).reset_index(drop=True)
    length = len(series)
    trend = [None] * length
    
    # Przypadek 1: okno nieparzyste
    if window % 2 == 1:
        
        half_window = (window - 1) // 2
        
        for i in range(length):
            
            if i < half_window or i >= length - half_window:
                trend[i] = np.nan
            
            else:
                trend[i] = series.iloc[
                    i - half_window : i + half_window + 1
                ].mean()
    
    # Przypadek 2: okno parzyste
    else:
        
        moving_averages = []
        
        # Zwykłe średnie ruchome MA
        for i in range(length - window + 1):
            avg = series.iloc[i : i + window].mean()
            moving_averages.append(avg)
        
        # CMA = średnia z dwóch sąsiednich MA
        for i in range(len(moving_averages) - 1):
            cma = (moving_averages[i] + moving_averages[i + 1]) / 2
            
            # np. dla window=4 pierwsza CMA trafia na indeks 2
            trend_index = i + window // 2
            
            trend[trend_index] = cma
    
    return trend


def additive_decomposition(df, value_col, period, window):
    """
    Ręczna klasyczna dekompozycja addytywna:
    
    Yt = Tt + St + Rt
    """
    
    result = df.copy()
    series = result[value_col].reset_index(drop=True)
    length = len(series)
    
    # 1. Trend
    trend = calculate_trend(series, window)
    result["Trend"] = trend
    
    # 2. Usunięcie trendu: Yt - Tt
    detrended = []
    
    for i in range(length):
        
        if pd.isna(result["Trend"].iloc[i]):
            detrended.append(np.nan)
        
        else:
            detrended.append(
                series.iloc[i] - result["Trend"].iloc[i]
            )
    
    result["Y_minus_T"] = detrended
    
    # 3. Numer sezonu
    seasons = []
    
    for i in range(length):
        seasons.append(i % period)
    
    result["Season"] = seasons
    
    # 4. Grupowanie i surowe wskaźniki sezonowości
    seasonality_raw = {}
    
    for season in range(period):
        values = result.loc[
            result["Season"] == season,
            "Y_minus_T"
        ].dropna()
        
        seasonality_raw[season] = values.mean()
    
    # 5. Normalizacja sezonowości
    seasonality_mean = np.mean(list(seasonality_raw.values()))
    
    seasonality_normalized = {}
    
    for season in seasonality_raw:
        seasonality_normalized[season] = (
            seasonality_raw[season] - seasonality_mean
        )
    
    # Przypisanie sezonowości do każdej obserwacji
    seasonality_all = []
    
    for i in range(length):
        season = result["Season"].iloc[i]
        seasonality_all.append(
            seasonality_normalized[season]
        )
    
    result["Seasonality"] = seasonality_all
    
    # 6. Składnik losowy: Rt = Yt - Tt - St
    residuals = []
    
    for i in range(length):
        
        if pd.isna(result["Trend"].iloc[i]):
            residuals.append(np.nan)
        
        else:
            residuals.append(
                series.iloc[i]
                - result["Trend"].iloc[i]
                - result["Seasonality"].iloc[i]
            )
    
    result["Residual"] = residuals
    
    return result, seasonality_raw, seasonality_normalized


def plot_decomposition(result, value_col):
    """
    Proste wykresy dekompozycji.
    """
    
    plt.figure(figsize=(10, 4))
    plt.plot(result[value_col], marker="o", label="Original")
    plt.plot(result["Trend"], marker="o", label="Trend")
    plt.title("Original series and trend")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    plt.figure(figsize=(10, 4))
    plt.plot(result["Seasonality"], marker="o", label="Seasonality")
    plt.title("Seasonality")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    plt.figure(figsize=(10, 4))
    plt.plot(result["Residual"], marker="o", label="Residual")
    plt.title("Residuals")
    plt.legend()
    plt.grid(True)
    plt.show()

## 3. Przykład 1 — okno nieparzyste, zwykłe MA

Tutaj ustawiamy:
- `period = 3`,
- `window = 3`.

Okno jest nieparzyste, więc trend można przypisać bezpośrednio do środka okna.

In [ ]:
data_odd = {
    "Time": range(1, 16),
    "Sales": [
        100, 108, 117,
        106, 114, 123,
        112, 120, 129,
        118, 126, 135,
        124, 132, 141
    ]
}

df_odd = pd.DataFrame(data_odd)
df_odd

In [ ]:
result_odd, raw_season_odd, norm_season_odd = additive_decomposition(
    df=df_odd,
    value_col="Sales",
    period=3,
    window=3
)

result_odd

In [ ]:
print("Surowe wskaźniki sezonowości:")
print(raw_season_odd)

print("\nZnormalizowane wskaźniki sezonowości:")
print(norm_season_odd)

print("\nSuma znormalizowanych sezonowości:")
print(sum(norm_season_odd.values()))

In [ ]:
plot_decomposition(result_odd, "Sales")

## 4. Przykład 2 — okno parzyste, CMA

Tutaj ustawiamy:
- `period = 4`,
- `window = 4`.

Okno jest parzyste, więc najpierw liczymy zwykłe MA, a potem CMA jako średnią z dwóch sąsiednich MA.

In [ ]:
data_even = {
    "Quarter": [
        "2020Q1", "2020Q2", "2020Q3", "2020Q4",
        "2021Q1", "2021Q2", "2021Q3", "2021Q4",
        "2022Q1", "2022Q2", "2022Q3", "2022Q4"
    ],
    "Sales": [
        95, 105, 120, 110,
        115, 125, 140, 130,
        135, 145, 160, 150
    ]
}

df_even = pd.DataFrame(data_even)
df_even

In [ ]:
result_even, raw_season_even, norm_season_even = additive_decomposition(
    df=df_even,
    value_col="Sales",
    period=4,
    window=4
)

result_even

In [ ]:
print("Surowe wskaźniki sezonowości:")
print(raw_season_even)

print("\nZnormalizowane wskaźniki sezonowości:")
print(norm_season_even)

print("\nSuma znormalizowanych sezonowości:")
print(sum(norm_season_even.values()))

In [ ]:
plot_decomposition(result_even, "Sales")

## 5. Jak czytać wynik?

Kolumny w tabeli oznaczają:

- `Sales` — oryginalny szereg, czyli $Y_t$,
- `Trend` — trend, czyli $T_t$,
- `Y_minus_T` — szereg po usunięciu trendu,
- `Season` — numer sezonu, np. kwartał,
- `Seasonality` — efekt sezonowy, czyli $S_t$,
- `Residual` — składnik losowy, czyli $R_t$.

Dla dekompozycji addytywnej zachodzi:

$$Y_t = T_t + S_t + R_t$$

czyli:

$$R_t = Y_t - T_t - S_t$$